# 02: Modeling

## Preliminares

In [1]:
# importar librerias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.functions import *
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

In [2]:
# Abrir ficheros de datos limpios
clean_df = pd.read_csv('data/merged_df_clean.csv')
y_full = pd.read_csv('data/y_full.csv')
ids_competencia = pd.read_csv('data/ids_test.csv')
with open('data/len_train.txt', 'r') as f:
    len_train = int(f.read())

clean_df.shape, y_full.shape, ids_competencia.shape, len_train

((2919, 75), (1460, 2), (1459, 1), 1460)

In [3]:
# Se crean listas de variables por tipo

# Numéricas
columnas_numericas = clean_df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Columnas numéricas: {len(columnas_numericas)}")

# Categóricas
columnas_categoricas = clean_df.select_dtypes(include=['object', 'category', 'str']).columns.tolist()
print(f"\nColumnas categóricas: {len(columnas_categoricas)}")

# Muestreo Estratificado: Luego de la depuración, se separa el test de la competencia del train original
X_competencia = clean_df.iloc[len_train:].copy()
X = clean_df.iloc[:len_train].copy()

# Particionar training test
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y_full['SalePriceLog'],
                                                    test_size=0.2,
                                                    random_state=42
                                                    )
# Se inicializa el registro de métricas
log_resultados = []

Columnas numéricas: 32

Columnas categóricas: 43


Se aplica Random Forest Regressor como primer modelo:

In [4]:
# Pipeline modelo1
nombre_modelo1 = 'Random Forest'

# Definir el preprocesado
preprocesado_rf = ColumnTransformer(
    transformers=[
        # Las numéricas pasan tal cual en el RF
        ('num', 'passthrough', columnas_numericas),

        # One-Hot Encoder para categoricas (crea pocas columnas nuevas)
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), columnas_categoricas)
    ]
)

# Crear el pipeline
pipe_rf = Pipeline([
    ('pre', preprocesado_rf),
    ('rf', RandomForestRegressor(n_estimators=300,
                                  random_state=42
                                  ))
])
# Entrenamiento
pipe_rf.fit(X_train, y_train)

# Predicción
y_pred = pipe_rf.predict(X_test)

# Se calculan las métricas y se guardan en log
log_resultados.append(obtener_metricas(y_test, y_pred, nombre_modelo1))
print("Entrenamiento finalizado.")
print("\nMétricas:", log_resultados[-1]) # se muestra siempre el último modelo en ejecutar

Entrenamiento finalizado.

Métricas: {'Modelo': 'Random Forest', 'MAE': 0.09716423662982576, 'MSE': 0.020765465251405615, 'RMSE': np.float64(0.14410227358166705), 'R2': 0.8887232450328159}


Se predice sobre los valores de la competencia y se crea el fichero de envío: 

In [5]:
# Predecir sobre los valores de la competencia
y_pred_competencia = pipe_rf.predict(X_competencia)
# Se pasan de escala logarítmica a dólares normales
precios_finales = np.expm1(y_pred_competencia)
# Aseguramos de que todo sea plano 
ids_planos = np.ravel(ids_competencia) 
precios_planos = np.ravel(precios_finales)
# Se exporta
salida = pd.DataFrame({
    'Id': ids_planos, 
    'SalePrice': precios_planos
})
salida.to_csv('data/submission.csv', index=False)